### Tools

Models can request to call tools that performs task such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    2. A function or coroutine to execute.

In [1]:
import os
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a common question about animal behavior, specifically avian vocalization and communication.\n\n2.  **Identify Key Concepts**:\n   - Parrots don\'t "talk" in the human sense (they don\'t understand language the way we do)\n   - They mimic sounds/vocalizations\n   - Evolutionary/biological reasons for mimicry\n   - Social/behavioral functions\n   - Brain structure (specialized vocal learning areas)\n   - Domestication/captive environment influence\n\n3.  **Core Scientific Facts**:\n   - Parrots are vocal learners (one of few bird groups that can learn sounds, like songbirds, hummingbirds, and corvids)\n   - In the wild, mimicry helps with social bonding, flock cohesion, predator avoidance, camouflage, and territory defense\n   - Their brain has specialized areas (song system) similar to humans in structure/function for vocal learning\n   - They assoc

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""   
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [7]:
response=model_with_tools.invoke("Whats the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user is asking for the current weather in Boston.\n2.  Identify available tools: I have a `get_weather` tool that takes a `location` parameter.\n3.  Extract parameters: Location = "Boston".\n4.  Call the tool: `get_weather(location="Boston")`.\n5.  Formulate response based on tool output. (I will simulate the tool call first) -> wait, I need to actually call it.\nLet\'s call the function.✅\n', 'tool_calls': [{'id': '3ggeb29mj', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 274, 'total_tokens': 418, 'completion_time': 0.275253821, 'completion_tokens_details': {'reasoning_tokens': 116}, 'prompt_time': 0.019232015, 'prompt_tokens_details': None, 'queue_time': 0.051094784, 'total_time': 0.294485836}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': '

### Tool Execution Loops

In [ ]:
#Step 1: Model generates tool calls
messages =[{"role":"user","content":"Whats the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with geneated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

Currently in Boston, it is experiencing **light drizzle**. The temperature is **20°C** (feels like 22°C), with high humidity at 94% and winds blowing at 10 km/h.


In [14]:
messages

[{'role': 'user', 'content': 'Whats the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: The user wants to know the weather in Boston.\n2.  Identify the available tool: `get_weather` which takes a `location` parameter.\n3.  Extract the location: "Boston".\n4.  Call the tool: `get_weather(location="Boston")`.\n5.  Formulate the response based on the tool\'s output. (I will simulate the tool call first).\nLet\'s call the tool. \nWait, I need to output the tool call exactly as specified.\nParameters: {"location": "Boston"}\nFunction: get_weather\nProceed. \nOutput matches the function calling format.✅\nNo extra text. Just the tool call. \nThen I will wait for the result, but since I\'m generating the response, I will just output the tool call.\nActually, I am the AI, I should generate the tool call.\nDone. \nProceeding. \n`print(get_weather(location="Boston"))` -> wait, the format is JSON-like in the

### Tool to fetch current weather condition for any location using Weatherstack API

In [ ]:
import os
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv()

@tool
def get_weather(location: str) -> str:
    """Fetch current real-time weather information for a given city or location using Weatherstack."""
    api_key = os.getenv("WEATHERSTACK_API_KEY")
    if not api_key:
        return "Error: WEATHERSTACK_API_KEY is not set in environment variables."

    # Weatherstack free tier uses HTTP endpoint
    url = "http://api.weatherstack.com/current"
    params = {
        "access_key": api_key,
        "query": location,
        "units": "m"  # 'm' for Metric (Celsius), 'f' for Fahrenheit
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if "error" in data:
            return f"Weatherstack API Error: {data['error'].get('info', 'Unknown error')}"

        current = data.get("current", {})
        loc = data.get("location", {})

        city = loc.get("name", location)
        country = loc.get("country", "")
        temp = current.get("temperature")
        feels_like = current.get("feelslike")
        weather_desc = ", ".join(current.get("weather_descriptions", []))
        humidity = current.get("humidity")
        wind_speed = current.get("wind_speed")

        return (
            f"Current Weather in {city}, {country}:\n"
            f"- Condition: {weather_desc}\n"
            f"- Temperature: {temp}°C (Feels like {feels_like}°C)\n"
            f"- Humidity: {humidity}%\n"
            f"- Wind Speed: {wind_speed} km/h"
        )
    except Exception as e:
        return f"Error fetching weather data: {str(e)}"


# Direct tool test
result = get_weather.invoke({"location": "Mumbai"})
print(result)


Current Weather in Mumbai, India:
- Condition: Light rain shower
- Temperature: 28°C (Feels like 32°C)
- Humidity: 79%
- Wind Speed: 25 km/h


In [ ]:
from langchain.agents import create_agent

# Create an agent equipped with your real-time weather tool
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",  # or "openai:gpt-4o-mini", "groq:llama-3.3-70b-versatile"
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use the weather tool whenever asked about current weather conditions."
)

# Run the agent
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather currently like in Mumbai, and should I carry an umbrella?"}]
})

print(response["messages"][-1].content)


[{'type': 'text', 'text': 'The current weather in Mumbai features light rain showers with a temperature of around 28°C (feeling like 32°C) and high humidity. \n\n**Yes, you should definitely carry an umbrella** since light rain showers are currently occurring and could persist.', 'extras': {'signature': 'El4KXAERTTIPbk+SOtaWmitmkOUAVGj9VWfvmIo4kmFSmL7ZZX5Hr9sy3zvIFAAiGZTZ8ptOtoK0UohLJdTlWTh87+PxjGgHXb00EvcNQ0NBYmT2z6SMfk9NNyxVbVHr'}}]
